# Продвинутый Python, семинар 14

**Лектор:** Петров Тимур

**Семинаристы:** Бузаев Федор, Дешеулин Олег, Коган Александра, Васина Олеся, Садуллаев Музаффар

## PDB и как его курить...

Debugger (дебаггер)
* инструмент для поиска и исправления ошибок в коде
* может иметь оболочку в виде интерфейса
* может иметь различную реализацию

Основные реализации:
* pdb (то что мы с вами рассмотрим)
* ipdb (Jupiter Notebook)
* rpdb (удаленная отладка)

PDB

* Python built-in debugger
* конкретная реализация
* текстовый отладчик
* не имеет программного интерфейса, исполняется в терминале

Давайте напишем какую-нибудь программу явную с багами и будем смотреть стектрейс...

In [70]:
%%writefile good_prog.py

def calc_sum(l):
    sum = 0
    for elem in l:
        sum = sum + elem

    return sum

calc_sum([1, 2, 3, 4])

Overwriting good_prog.py


In [72]:
%%writefile zero_division.py

def compute_average(values):
    total = 0
    for v in values:
        total += v

    breakpoint()   # Stop before error
    avg = total / len(values)
    return avg

compute_average([])
print('Here')

Overwriting zero_division.py


Мы написали "случайный код". Сохранили в файлик. Давайте его будем исследовать. Для того чтобы его вызвать нужно призвать команду. \
 `<ваш инстанс используемого питона> -m pdb <название_py_модуля>.py`

***API***

Наше API которым можно пользоваться (основное):
* break()
Делать брекпоинты при этом не трогая руками искомый код!

- `list` или `l`
- `list first`
- `list first, second`


Покажет часть кода, на выполнении которой сейчас находится интерпретатор. Можно передать два аргумента `first` и `last` для просмотра определённого участка кода.
Если указывать только `first`, то будет выведен код вокруг искомой строки.

* `up p` и `down d`

Передвижения по стеку вызовов. Cмотрим откуда была вызвана текущая функция.

* `step` или `s`; `next` или `n`

С их помощью будем работать построчно. Единственное различие между ними в том, что `next` перейдёт к следующей строке вне зависимости от вызываемых функций, а `step` перейдёт в вызванную функцию, если это возможно.

* `print var` или `p var`

Печать значения переменной в процессе исполнения

* `breakpoint var` или `b var`; `b line_number`; `b function_name`; `b file_name:line_number`

Команда останова на конкретной строке

* `cont` или `c`

Продолжить выполнение программы до следующего breakpoint

* `where`

Посмотреть весь стектрейс

* `quit` или `q`

Завершить debugger

`!python3 -m pdb good_prog.py` - необходимо выполнить в терминале, тут работать не будет или не гарантируется.

`!python3 -m pdb zero_division.py`

## Интерпретатор Python

### Интерпретатор v.s. компилятор

***Интерпретатор*** — это программа, которая читает исходный код и выполняет его построчно, сразу, без отдельного шага генерации исполняемого файла.

Основные свойства интерпретатора:

- Выполнение кода сразу, без отдельной сборки.

- Проще отлаживать — можно остановиться в любой момент (pdb).

- Чаще медленнее, чем компилируемые языки.

***Компилятор*** — это программа, которая сначала полностью преобразует исходный код в другой язык, обычно в машинный код, и только после этого программа запускается.

Основные свойства компилятора

- Программа заранее переводится в исполняемый файл.

- Запуск очень быстрый.

- Хорошая оптимизация.

Но:

- нельзя просто “вставить код и выполнить сразу”

- ошибки видны до запуска программы (то есть во время компиляции), а не во время.

***Сравнение интерпретатора и компилятора***

| Характеристика              | Интерпретатор                        | Компилятор       |
| --------------------------- | ------------------------------------ | ---------------- |
| **Когда выполняется код?**  | сразу                                | после компиляции |
| **Нужен исполняемый файл?** | нет                                  | да               |
| **Скорость запуска**        | быстрая                              | медленнее        |
| **Скорость выполнения**     | медленнее                            | быстрее          |
| **Отладка**                 | простая, можно “входить” в программу | сложнее          |
| **Типичные языки**          | Python, JavaScript, PHP, Ruby        | C, C++, Rust, Go |


### Поговорим про интерпретатор Python

Когда мы запускаем python script.py, Python делает:



```
Читает текст (source code)
↓
Токенизация (tokenize)
↓
Построение AST (ast.parse)
↓
Компиляция в байткод (compile)
↓
Исполнение байткода виртуальной машиной CPython
```


Рассмотрим весь цикл работы программмы

In [17]:
program = """\
x = 1
y = 2
print(3*(x+y))
"""

In [19]:
exec(program) # выполнить код в тексте

9


Как интерпретатор ее выполняет?

#### Шаг 1: Токенизация

In [32]:
import io
import tokenize


tokens = tokenize.generate_tokens(io.StringIO(program).readline)
list(tokens)

[TokenInfo(type=1 (NAME), string='x', start=(1, 0), end=(1, 1), line='x = 1\n'),
 TokenInfo(type=55 (OP), string='=', start=(1, 2), end=(1, 3), line='x = 1\n'),
 TokenInfo(type=2 (NUMBER), string='1', start=(1, 4), end=(1, 5), line='x = 1\n'),
 TokenInfo(type=4 (NEWLINE), string='\n', start=(1, 5), end=(1, 6), line='x = 1\n'),
 TokenInfo(type=1 (NAME), string='y', start=(2, 0), end=(2, 1), line='y = 2\n'),
 TokenInfo(type=55 (OP), string='=', start=(2, 2), end=(2, 3), line='y = 2\n'),
 TokenInfo(type=2 (NUMBER), string='2', start=(2, 4), end=(2, 5), line='y = 2\n'),
 TokenInfo(type=4 (NEWLINE), string='\n', start=(2, 5), end=(2, 6), line='y = 2\n'),
 TokenInfo(type=1 (NAME), string='print', start=(3, 0), end=(3, 5), line='print(3*(x+y))\n'),
 TokenInfo(type=55 (OP), string='(', start=(3, 5), end=(3, 6), line='print(3*(x+y))\n'),
 TokenInfo(type=2 (NUMBER), string='3', start=(3, 6), end=(3, 7), line='print(3*(x+y))\n'),
 TokenInfo(type=55 (OP), string='*', start=(3, 7), end=(3, 8), line

Каждый элемент — это `TokenInfo` с полями:

```
TokenInfo(
    type=...,    # тип токена (числовой код)
    string=...,  # текст токена
    start=...,   # (строка, колонка начала)
    end=...,     # (строка, колонка конца)
    line=...,    # исходная строка целиком
)
```

In [29]:
tokenize.TokenInfo(type=1, string='x', start=(1, 0), end=(1, 1), line='x = 1\n')

TokenInfo(type=1 (NAME), string='x', start=(1, 0), end=(1, 1), line='x = 1\n')


```
string='x' — текст токена.

type=1 — тип (1 = NAME, имя переменной/функции и т.д.).

start=(1, 0) — строка 1, позиция 0 (начало 'x').

end=(1, 1) — строка 1, позиция 1 (конец 'x').

line='x = 1\n' — вся строка, из которой этот токен взят.
```



**Зачем нужен io.StringIO(program).readline**

`tokenize.generate_tokens` ожидает функцию, которая читает по одной строке (как у файлов — `file.readline`).


**Обычно**:

```

with open("script.py", "r", encoding="utf-8") as f:
    tokens = tokenize.generate_tokens(f.readline)
```

Но у нас код уже в строке, файла нет.
Поэтому:


```
io.StringIO(program)  # создаёт "файл в памяти"
io.StringIO(program).readline  # даёт функцию, которая читает строки

```



In [36]:
import io
import tokenize

program = """\
x = 1
y = 2
print(3*(x+y))
"""

for tok in tokenize.generate_tokens(io.StringIO(program).readline):
    tok_type = tokenize.tok_name[tok.type]
    print(f"{tok_type:<10} {tok.string!r}")


NAME       'x'
OP         '='
NUMBER     '1'
NEWLINE    '\n'
NAME       'y'
OP         '='
NUMBER     '2'
NEWLINE    '\n'
NAME       'print'
OP         '('
NUMBER     '3'
OP         '*'
OP         '('
NAME       'x'
OP         '+'
NAME       'y'
OP         ')'
OP         ')'
NEWLINE    '\n'
ENDMARKER  ''


Какие бывают типы токенов

- NAME — имена: x, y, print

- NUMBER — числа: 1, 2, 3

- OP — операторы/символы: =, +, *, (, )

- NEWLINE — конец строки

- INDENT / DEDENT — отступы (при if, for, функциях и т.п.)

- STRING — строковые литералы: "hello", 'abc'

- ENDMARKER — конец файла

- ...

**Токены vs AST (кратко)**

- tokenize смотрит на код как на поток символов, деля его на кусочки (токены).

- ast (модуль ast) строит синтаксическое дерево — структурированное представление программы.

#### Шаг 2: Построение абстрактного синтаксического дерева



Грамматика языка: https://docs.python.org/3/reference/grammar.html

In [45]:
!pip install astpretty



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [46]:
import ast
import astpretty

In [47]:
program = """\
x = 1
y = 2
print(3*(x+y))
"""

tree = ast.parse(program)
print(ast.dump(tree, indent=4))

Module(
    body=[
        Assign(
            targets=[
                Name(id='x', ctx=Store())],
            value=Constant(value=1)),
        Assign(
            targets=[
                Name(id='y', ctx=Store())],
            value=Constant(value=2)),
        Expr(
            value=Call(
                func=Name(id='print', ctx=Load()),
                args=[
                    BinOp(
                        left=Constant(value=3),
                        op=Mult(),
                        right=BinOp(
                            left=Name(id='x', ctx=Load()),
                            op=Add(),
                            right=Name(id='y', ctx=Load())))],
                keywords=[]))],
    type_ignores=[])


Разберём, что тут напечаталось

Корень дерева Module

```
Module(
    body=[ ...список операторов... ]
)

```


`body` — список инструкций верхнего уровня.

В нашем случае их три:

`x = 1 → Assign`

`y = 2 → Assign`

`print(3*(x + y)) → Expr`


Присваивания `x = 1`

```
Assign(
    targets=[Name(id='x', ctx=Store())],
    value=Constant(value=1)
)
```

- `Assign` — узел "оператор присваивания".

- `targets` — список того, кому присваиваем (обычно один элемент).

- `Name(id='x', ctx=Store())`:

    - `Name` — узел имени.

    - `id='x'` — сама строка имени.

    - `ctx=Store()` — контекст: “мы записываем сюда значение”.

    - `value=Constant(value=1)` — значение справа от =.

Аналогично для `y = 2`.

Вызов print: `print(3*(x+y))`

```
Expr(
    value=Call(
        func=Name(id='print', ctx=Load()),
        args=[ ... ],
        keywords=[]
    )
)
```

`Expr` — просто "выражение как отдельная строка" (когда мы что-то вызываем или пишем без =, if и т.п.).

- `Call` — вызов функции.

- `func=Name(id='print', ctx=Load())`:

    - `Name` — имя.

    - `ctx=Load()` — “мы читаем имя”, используем его значение.

    - `args` — позиционные аргументы функции (список).

    - `keywords` — именованные аргументы типа `print(x=...)` (у нас пусто).



```
 BinOp(
    left=Constant(value=3),
    op=Mult(),
    right=BinOp(
        left=Name(id='x', ctx=Load()),
        op=Add(),
        right=Name(id='y', ctx=Load())
    )
)

```

Это дерево умножения:

- Внешний `BinOp (3 * (...))`:

  - `left = Constant(3)`

  - `op = Mult()` — оператор *

  - `right = BinOp(...)` — это внутри скобок.

- Внутренний `BinOp (x + y)`:

  - `left = Name('x', Load)`

  - `op = Add()` — оператор +

  - `right = Name('y', Load)`

Важно:

AST “понимает” структуру выражения — что к чему относится, какая операция над чем.

Можно обойти дерево и что-то с ним сделать. Например, найти все имена:

In [ ]:
import ast

program = """\
x = 1
y = 2
print(3*(x+y))
"""

tree = ast.parse(program)

names = set()

for node in ast.walk(tree):
    if isinstance(node, ast.Name):
        names.add(node.id)

print(names)

{'print', 'y', 'x'}


**Сравнение: tokenize vs AST**

tokenize:

- видит код как последовательность “кусочков текста” (токены).

- не знает, что x + y — это операция над чем-то; видит NAME, OP, NAME.

ast:

- видит код как структуру:

 - присваивания,

 - вызовы функций,

 - бинарные операции,

 - имена, числа, строки и т.д.

 - подходит для “понимания” кода: анализ, рефакторинг, переписывание.

#### Шаг 3: Компиляция в байткод

Что такое ***байткод Python***

Это низкоуровневые инструкции, которые выполняет интерпретатор.

Пример байткода:

```
LOAD_NAME                0 (x)
LOAD_CONST               1 (1)
BINARY_ADD
STORE_NAME               0 (x)

```



Это не машинный код, а промежуточный язык, понятный виртуальной машине Python.

Python НЕ выполняет код напрямую. Происходит цепочка:

- Парсер → превращает текст в AST (абстрактное синтаксическое дерево)

- Компилятор → преобразует AST в байткод

- Интерпретатор (виртуальная машина) → выполняет байткод

- Байткод — это инструкции для виртуальной машины Python (CPython VM).

In [ ]:
program = "x = 1 + 2"
code = compile(program, "<string>", "exec")
print(code)

<code object <module> at 0x114521e40, file "<string>", line 1>
  0           0 RESUME                   0

  1           2 LOAD_CONST               0 (3)
              4 STORE_NAME               0 (x)
              6 RETURN_CONST             1 (None)


In [55]:
code

<code object <module> at 0x114521e40, file "<string>", line 1>

code — это специальный объект code object, содержащий байткод, константы, имена переменных и т. д.

***Где Python хранит байткод (.pyc файлы)***

Файл `.pyc` содержит:

- кэш байткода,

- версию Python,

- контрольную сумму исходника.

Это позволяет запускать программу быстрее при повторных запусках.

Важно: `__pycache__` директория создаётся только при импорте py модулей, при непосредственном запуске py скриптов весь байт-код хранится в ОЗУ


In [64]:
%%writefile script.py

x = 1
y = 2
z = x + y

Overwriting script.py


In [65]:
import script

print(script.z)

3


Как прочитать .pyc файл

In [66]:
import marshal

In [67]:
with open("__pycache__/script.cpython-312.pyc", "rb") as f:
    f.read(16)  # пропустить заголовок
    code = marshal.load(f)

In [68]:
import dis
dis.dis(code)

  0           0 RESUME                   0

  2           2 LOAD_CONST               0 (1)
              4 STORE_NAME               0 (x)

  3           6 LOAD_CONST               1 (2)
              8 STORE_NAME               1 (y)

  4          10 LOAD_NAME                0 (x)
             12 LOAD_NAME                1 (y)
             14 BINARY_OP                0 (+)
             18 STORE_NAME               2 (z)

  5          20 PUSH_NULL
             22 LOAD_NAME                3 (print)
             24 LOAD_NAME                2 (z)
             26 CALL                     1
             34 POP_TOP
             36 RETURN_CONST             2 (None)


In [69]:
def f(x):
    return x+1

list(dis.get_instructions(f))


[Instruction(opname='RESUME', opcode=151, arg=0, argval=0, argrepr='', offset=0, starts_line=1, is_jump_target=False, positions=Positions(lineno=1, end_lineno=1, col_offset=0, end_col_offset=0)),
 Instruction(opname='LOAD_FAST', opcode=124, arg=0, argval='x', argrepr='x', offset=2, starts_line=2, is_jump_target=False, positions=Positions(lineno=2, end_lineno=2, col_offset=11, end_col_offset=12)),
 Instruction(opname='LOAD_CONST', opcode=100, arg=1, argval=1, argrepr='1', offset=4, starts_line=None, is_jump_target=False, positions=Positions(lineno=2, end_lineno=2, col_offset=13, end_col_offset=14)),
 Instruction(opname='BINARY_OP', opcode=122, arg=0, argval=0, argrepr='+', offset=6, starts_line=None, is_jump_target=False, positions=Positions(lineno=2, end_lineno=2, col_offset=11, end_col_offset=14)),
 Instruction(opname='RETURN_VALUE', opcode=83, arg=None, argval=None, argrepr='', offset=10, starts_line=None, is_jump_target=False, positions=Positions(lineno=2, end_lineno=2, col_offset=4

Bytecode ≠ машинный код

Байткод НЕ тот код, который выполняет процессор.
Выполняет его виртуальная машина CPython, которая интерпретирует инструкции.

Пример инструкции VM:

- LOAD_NAME

- LOAD_CONST

- CALL_FUNCTION

- JUMP_IF_FALSE_OR_POP

- RETURN_VALUE

Это примерно как P-code в Pascal, JVM-байткод в Java или байткод Lua.

Зачем вообще нужен байткод?


- Ускорение исполнения

Парсинг текста — дорогая операция.
Скомпилированный байткод можно просто переиспользовать.

- Портируемость

Один и тот же байткод работает на любой платформе (Windows, Linux, Mac).

- Безопасность и оптимизация

Компилятор знает, какие имена нужны, какие переменные локальные, какие глобальные, и может оптимизировать константы.